<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*



**Research question:**
Can search-performance signals available in the first half of a month help prioritize which content pages should be reviewed first for possible future decline?

The project treats this as a **ranking problem**. Each eligible page receives a risk score so that analysts can review higher-risk pages first when review capacity is limited.

The model is used for decision support only:

**Model prioritizes → human investigates → human decides.**

It does not automatically recommend rewriting, deleting, or publishing content, and it does not attempt to predict Google's ranking algorithm.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*



The analysis uses the **FlyRank ML Internship warehouse release, build `v20260703`**.

The main table used is:

`fact_content_daily_performance`

The full warehouse contains approximately **78.8 million daily performance rows**, but this experiment uses only the **March 2026 partition**.

### Date windows

* **Feature window:** March 1–15, 2026
* **Outcome window:** March 16–31, 2026

The final modeling cohort contains **61,795 content pages across 34 pseudonymized clients**.

Pages were included only when Google Search Console data was available for the complete feature and outcome windows.

### Exclusions

The model does not use:

* `client_id`
* `content_id`
* future impression fields
* future average daily impressions
* `impression_change_pct`
* `is_declining_proxy`

Client and content IDs are used only for grouping and validation.

Future and target-related fields are excluded to prevent **data leakage**, because they would not have been available at the prediction point.

The analysis uses only pseudonymized data. No client names, URLs, private queries, or access credentials are included.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*



The task is treated as a **ranking problem**. Each eligible page receives a risk score, and higher-risk pages are reviewed first.

### Assumption

Search-performance patterns observed during **March 1–15, 2026** may contain useful signals for identifying pages that are more likely to experience a meaningful decline during **March 16–31, 2026**.

This is a decision-support assumption, not a causal claim.

### Features

The final model uses **21 pre-outcome features** grouped into four categories:

* **Momentum:** recent impression movement
* **Current performance:** impressions, clicks, CTR, average position, and activity rate
* **Trend shape:** changes in impressions, clicks, CTR, position, and recent trajectory
* **Client-relative context:** how a page compares with other pages from the same pseudonymized client

All features are calculated using only data available before the outcome window.

### Label definition

A page is labeled as a future decline case when its average daily impressions during March 16–31 are more than **20% lower** than its average daily impressions during March 1–15.

This target is a **future-impression-decline proxy** and is not intended to represent a universal definition of content decay.

### Baseline

The Week-4 rule-based baseline prioritizes pages when:

1. average search position is **20 or better**, and
2. CTR is low relative to pages in the same search-position group.

Matching pages are ranked using their first-half impressions.

### Model

The primary model is a **Random Forest classifier** using the full 21-feature set.

A second **momentum-only Random Forest** using two impression-momentum features is included as a stronger comparison.

### Validation design

The final evaluation uses **5-fold client-disjoint grouped validation**.

Complete clients are held out from training in each fold, so pages from the same client do not appear in both training and validation.

The primary evaluation metric is **Precision@100**, because the practical goal is to rank useful review candidates near the top of a limited review queue.

### Leakage checks

The final feature set was audited to exclude:

* client and content IDs from model inputs
* future impression fields
* future average impressions
* `impression_change_pct`
* `is_declining_proxy`
* any feature derived from the outcome period

The final audit found no forbidden overlap, no suspicious future-derived features, and no missing values in the final model feature set.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*



The primary comparison uses the same **5-fold client-disjoint validation design**, the same target, and the same **Precision@100** metric for all methods.

| Method                        | Mean Precision@100 |    SD |
| ----------------------------- | -----------------: | ----: |
| Random Forest — Full Signal   |          **74.6%** | 13.6% |
| Random Forest — Momentum Only |          **66.2%** | 12.8% |
| Week-4 Rule Baseline          |          **35.2%** |  6.4% |

The full-signal Random Forest achieved the strongest observed ranking performance.

Compared with the Week-4 rule baseline, it improved mean Precision@100 by **39.4 percentage points**.

Compared with the stronger momentum-only Random Forest, the improvement was smaller at **8.4 percentage points**.

This second comparison is important because it suggests that the additional trend-shape, CTR, position, performance-level, and client-relative features provide useful ranking information beyond recent impression momentum alone.

The final result is lower than the earlier **90.0% Precision@100** obtained with random-row cross-validation. Under the stricter client-disjoint validation, performance was **74.6%**.

This drop is expected because complete clients are held out from training. The **74.6% client-disjoint result is therefore the primary performance number reported in this study**.

The final March review queue contains **61,795 pages across 34 pseudonymized clients**, with an observed decline rate of **32.4%**.

These results support the model as a **decision-support ranking tool**, but they do not show that the model will perform equally well for every client or future time period.


## 5. Limitations

*What this work cannot claim.*

This study has several important limitations.

First, the target is a **future-impression-decline proxy** based on a greater than 20% reduction in average daily impressions. It is useful for evaluation, but it is not a universal definition of content decay or content quality.

Second, the primary experiment uses **March 2026** as the main evaluation period. Performance may change across other months, clients, or market conditions.

Third, the dataset contains clients with different traffic levels, content volumes, and history depth. Although client-disjoint validation reduces leakage from client-specific patterns, it does not guarantee identical generalization to every new client.

Fourth, fold-level variation remains substantial. The full model achieved **74.6% mean Precision@100 with a 13.6% standard deviation**, which shows that performance is not equally strong across all held-out client groups.

Fifth, the model identifies statistical patterns in pre-outcome search-performance signals. It does **not** establish that any feature causes future decline.

The model also does not predict Google's ranking algorithm and does not prove that refreshing, rewriting, or otherwise changing a recommended page will improve future performance.

Additional **temporal and prospective validation** would be required before making stronger production-level claims.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*



The model output is converted into a **ranked human-review queue**.

The operating policy uses the within-fold model risk percentile:

* **Risk percentile ≥ 0.80:** active human review
* **0.50 ≤ risk percentile < 0.80:** watchlist
* **Risk percentile < 0.50:** monitor

For high-risk pages, observable pre-outcome signals are used to assign a review type.

| Observed pattern                                      | Recommended action            |
| ----------------------------------------------------- | ----------------------------- |
| Sustained decline pattern                             | `CONTENT_REFRESH_REVIEW`      |
| Ranking slippage                                      | `SERP_AND_INTENT_REVIEW`      |
| Visible page with relatively low CTR                  | `TITLE_META_CTR_REVIEW`       |
| Client-relative anomaly                               | `MANUAL_DIAGNOSTIC_REVIEW`    |
| High model risk without a clear heuristic explanation | `HUMAN_REVIEW_BEFORE_EDIT`    |
| Medium-risk page                                      | `WATCHLIST_NO_IMMEDIATE_EDIT` |
| Lower-risk page                                       | `MONITOR_NO_IMMEDIATE_EDIT`   |

These recommendations are **investigation priorities, not automatic editing instructions**.

For example, `CONTENT_REFRESH_REVIEW` means that an analyst should investigate whether a refresh may be appropriate. It does not mean that the page should automatically be rewritten.

The reason codes describe observable signals associated with the ranking. They are not causal explanations for why a page declined.

The operating principle is:

**Model prioritizes → human investigates → human decides.**


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*



The deployed research paper uses the following figures generated from the final deterministic evaluation.

### Figure 1 — Validation performance

`work/figures/01_validation_performance.png`

Compares the **Full-Signal Random Forest**, **Momentum-Only Random Forest**, and **Week-4 Rule Baseline** under the same client-disjoint validation design.

**Main takeaway:** the full model achieved the strongest observed Precision@100, while the momentum-only comparison shows that much of the predictive signal comes from recent momentum.

### Figure 2 — Cumulative capture curve

`work/figures/02_cumulative_capture_curve.png`

Shows how many observed decline cases are captured as a larger share of the ranked review queue is inspected.

**Main takeaway:** the ranking concentrates more decline cases near the top of the queue, supporting its use when analyst review capacity is limited.

### Figure 3 — Client generalization

`work/figures/04_client_generalization.png`

Shows model performance across held-out clients rather than only reporting one overall average.

**Main takeaway:** performance varies across clients, which supports using the model as decision support and motivates further validation before production use.

### Figure 4 — Risk and value matrix

`work/figures/05_risk_value_matrix.png`

Combines model risk with relative page exposure to support practical review prioritization.

**Main takeaway:** model risk determines review priority, while observable context helps analysts decide what type of investigation may be useful.

### Supporting figure

`work/figures/03_risk_decile_outcomes.png`

Shows how the observed decline rate changes across model-risk deciles.

This figure provides additional evidence that higher model scores correspond to higher observed decline rates, but it is secondary to the main validation and ranking figures.


--------------------------------------------------------------------------

## 5-Minute Demo Outline

### 1. Research Question

**Can search-performance signals available in the first half of a month help FlyRank prioritize which content pages should be reviewed first for possible future decline?**

The practical problem is limited review capacity. When a content portfolio contains thousands of pages, analysts cannot investigate everything at once. This project tests whether machine learning can produce a useful ranked review queue so that higher-risk pages are investigated first.

The workflow remains human-led:

**Model prioritizes → human investigates → human decides.**

---

### 2. Method

I used the **FlyRank ML Internship warehouse release, build `v20260703`**, focusing on the March 2026 partition.

The final modeling cohort contained:

* **61,795 content pages**
* **34 pseudonymized clients**
* Feature window: **March 1–15, 2026**
* Outcome window: **March 16–31, 2026**

The target identified pages whose average daily impressions declined by more than **20%** in the outcome period.

I compared three ranking approaches:

1. **Random Forest — Full Signal** using 21 pre-outcome features
2. **Random Forest — Momentum Only** using recent impression-momentum signals
3. **Week-4 Rule Baseline**

Evaluation used **5-fold client-disjoint validation**, meaning complete clients were held out during validation. This provides a more realistic test than randomly splitting pages from the same clients across training and validation.

The primary metric was **Precision@100**, because the practical question is whether the highest-ranked pages are useful candidates for limited analyst review.

---

### 3. One Chart to Show

**Show the model comparison chart for Precision@100.**

Use the chart to explain the main comparison:

| Method                        | Mean Precision@100 |
| ----------------------------- | -----------------: |
| Random Forest — Full Signal   |          **74.6%** |
| Random Forest — Momentum Only |          **66.2%** |
| Week-4 Rule Baseline          |          **35.2%** |

**Key message:** the full-signal model ranked future-decline cases substantially better than the original rule baseline, while the momentum-only model showed that recent performance movement already contains a strong signal.

---

### 4. One Honest Result

The strongest observed result was the **74.6% mean Precision@100** achieved by the full Random Forest under client-disjoint validation.

This was:

* **39.4 percentage points higher** than the Week-4 rule baseline
* **8.4 percentage points higher** than the stronger momentum-only model

The second comparison is important. It shows that the improvement is not simply because machine learning replaced a weak rule. Recent momentum alone was already competitive, while the broader feature set added further ranking value.

Performance also varied across held-out client groups, with a **13.6% standard deviation** for the full model. Therefore, the result should be treated as evidence that the approach is useful for prioritization, not as proof that it will perform equally well for every client or future period.

The model does **not** predict Google's ranking algorithm and does not prove that editing a recommended page will improve its future performance.

---

### 5. Recommendation

Use the model as a **decision-support ranking layer** for content review rather than an automated content-editing system.

A practical workflow would be:

**Rank pages by model risk → review the highest-risk pages first → inspect the underlying search signals → decide whether a content, SERP, CTR, or manual diagnostic review is justified.**

For production use, I would keep recent momentum as a strong benchmark and continue validating the full model across additional time periods and unseen clients before relying on it more broadly.

### Closing Message

The main takeaway is not that machine learning can automatically identify what content should be changed.

It is that, with careful leakage control, client-disjoint validation, and a strong baseline, search-performance signals can help turn a very large content portfolio into a more focused and defensible human review queue.


## Shareable Cuts

### Short Social Post

I completed my FlyRank ML capstone on **content decline prioritization**.

The project explored whether early search-performance signals can help prioritize which content pages should be reviewed first when analyst time is limited.

Using Python, SQL, DuckDB, and scikit-learn, I built a ranking workflow over **61,795 content pages across 34 pseudonymized clients**. I compared a full Random Forest model with a momentum-only model and a rule-based baseline, using **5-fold client-disjoint validation** to reduce leakage and test performance on unseen clients.

The full model achieved **74.6% mean Precision@100**, compared with **66.2%** for the momentum-only model and **35.2%** for the rule baseline.

The main lesson was that model choice is only part of the work. Strong baselines, leakage control, realistic validation, and honest interpretation are just as important.

The system is designed for decision support:

**Model prioritizes → human investigates → human decides.**

Research paper: https://dania-yasir.github.io/flyrak-project/

---

### Employer-Facing Summary

I built an end-to-end machine learning ranking workflow to help prioritize content pages for human review using the FlyRank ML Internship search-performance dataset. The final experiment evaluated **61,795 pages across 34 pseudonymized clients** using leakage-aware feature engineering and **5-fold client-disjoint validation**. The full Random Forest achieved **74.6% mean Precision@100**, outperforming both a momentum-only model at **66.2%** and the rule-based baseline at **35.2%**, while remaining a decision-support system rather than an automated content decision tool.

## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.